In [1]:
import pandas as pd
import numpy as np
import netCDF4 as nc

In [2]:
def extract_df(year, rcp, lon1, lon2, lat1, lat2):
    # Open the netCDF file
    name = f'POLCOMS_ERSEM_biogeochemical-monthly-all-rcp{rcp}-thetao-{year}-08-v1.1.nc'
    dataset = nc.Dataset(name)
    latitude = dataset.variables['lat'][:]
    longitude = dataset.variables['lon'][:]
    temperature = dataset.variables['thetao'][:]
    time = dataset.variables['time'][:]
    dataset.close()

    name = f'POLCOMS_ERSEM_biogeochemical-monthly-all-rcp{rcp}-uo-{year}-08-v1.1.nc'
    dataset = nc.Dataset(name)
    uo = dataset.variables['uo'][:]
    dataset.close()

    name = f'POLCOMS_ERSEM_biogeochemical-monthly-all-rcp{rcp}-vo-{year}-08-v1.1.nc'
    dataset = nc.Dataset(name)
    vo = dataset.variables['vo'][:]
    dataset.close()
    
    # Create a dataframe with longitude, latitude, and temperature values
    lon_grid, lat_grid = np.meshgrid(longitude, latitude)
    tem_df = pd.DataFrame({
        'lon': lon_grid.ravel(),
        'lat': lat_grid.ravel(),
        'east': uo[0, 0,:, :].ravel(),
        'north': vo[0, 0,:, :].ravel(),
        'value': temperature[0, 0,:, :].ravel()  # Assuming depth index 0 for temperature
    })
    
    # Filter the longitudes and latitudes within the specified range
    lon_sub = longitude[(longitude > lon1) & (longitude < lon2)]
    lat_sub = latitude[(latitude > lat1) & (latitude < lat2)]
    
    # Create a meshgrid of the filtered coordinates
    lon_grid_sub, lat_grid_sub = np.meshgrid(lon_sub, lat_sub)
    coords = np.vstack([lon_grid_sub.ravel(), lat_grid_sub.ravel()]).T
    
    # Find matching rows in the full dataframe
    temp = tem_df[(tem_df['lon'].isin(lon_sub)) & (tem_df['lat'].isin(lat_sub))]
    
    return temp

# Initialize an empty dataframe to store all the years
total = pd.DataFrame()

# Loop through the years and append the data for each year
for year in range(2006, 2023):
    df = extract_df(year, 45, 5,22,35,46)
    df['year'] = year
    df['RCP'] = 'rcp45'
    total = pd.concat([total, df], ignore_index=True)
    df = extract_df(year, 85, 5,22,35,46)
    df['year'] = year
    df['RCP'] = 'rcp85'
    total = pd.concat([total, df], ignore_index=True)
year = 2050
df = extract_df(year, 45, 5,22,35,46)
df['year'] = year
df['RCP'] = 'rcp45'
total = pd.concat([total, df], ignore_index=True)
df = extract_df(year, 85, 5,22,35,46)
df['year'] = year
df['RCP'] = 'rcp85'
total = pd.concat([total, df], ignore_index=True)

year = 2099
df = extract_df(year, 45, 5,22,35,46)
df['year'] = year
df['RCP'] = 'rcp45'
total = pd.concat([total, df], ignore_index=True)
df = extract_df(year, 85, 5,22,35,46)
df['year'] = year
df['RCP'] = 'rcp85'
total = pd.concat([total, df], ignore_index=True)

In [3]:
unique = total.drop_duplicates(subset=['year'], keep='first')
unique

,lon,lat,east,north,value,year,RCP
0,5.1,35.099998,NaN,NaN,NaN,2006,rcp45
36842,5.1,35.099998,NaN,NaN,NaN,2007,rcp45
73684,5.1,35.099998,NaN,NaN,NaN,2008,rcp45
110526,5.1,35.099998,NaN,NaN,NaN,2009,rcp45
147368,5.1,35.099998,NaN,NaN,NaN,2010,rcp45
184210,5.1,35.099998,NaN,NaN,NaN,2011,rcp45
221052,5.1,35.099998,NaN,NaN,NaN,2012,rcp45
257894,5.1,35.099998,NaN,NaN,NaN,2013,rcp45
294736,5.1,35.099998,NaN,NaN,NaN,2014,rcp45
331578,5.1,35.099998,NaN,NaN,NaN,2015,rcp45


In [4]:
save = total.to_csv('Projections_August.csv', index=False)

In [5]:
import os
import glob
folder_path = ''

# Find all files starting with 'polcoms' in the folder
files_to_delete = glob.glob(os.path.join(folder_path, 'POLCOMS*'))

# Delete each file
for file_path in files_to_delete:
    os.remove(file_path)